# OCO edge 회귀 반경 × 배율 비교
제품마다 기존 `27_OCO_FINAL_POINTS.parquet`와 같은 실행의 `extension_settings.json`만 필요합니다. 기존 output 폴더 또는 두 파일을 모아 둔 폴더를 00번에 지정하세요. **EP_BASE_DIR 기본 E:\PHOTO_ANALYSIS\BASIC 경로를 실제 BASIC 경로로 수정한 뒤 00→09 순서로 실행**합니다. DB 조회·기존 notebook 수정은 없습니다.

**KGVM·KHVF는 Y17 / PBVC(VC)는 X21.** K-MRC 고정 상태에서 edge 배율 10~100%(10% 간격), 회귀 반경 146/147mm의 20조건을 제품별 비교합니다. 배율은 PSM 추정에만 사용합니다. 평가는 원래 측정값에서 새 CPE를 뺀 residual로, 같은 ≤144mm / 전체 ≤147mm point에서 수행합니다. 회귀에서 뺀 146~147mm도 평가에는 남깁니다.

실제 wafer/shot별 edge point 비율과 같은 mixed shot의 안쪽·edge 입력 M3S 차이를 함께 저장합니다. Point가 부족해도 para 수는 줄이지 않고 기존 ridge를 적용하되 진단을 남깁니다. 학습 point 0개 등 예측 누락이 생기면 해당 평가 그룹 M3S는 NA입니다. 제품 곡선은 모든 조건에서 유효한 동일 wafer 집합을 사용합니다.

결과는 `output_edge_radius_sweep`에 저장됩니다. 제품별 기존 최적 배율 VM50/VF20/VC40은 00번 reference_scale에 기록하며, 100% 열화와 입력 차이의 관계를 보는 참고값으로만 사용합니다.


## 00. 제품별 입력·방향·반경·배율 설정


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import copy
import json
import os
from functools import lru_cache
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

EP_BASE_DIR = Path(r"E:\PHOTO_ANALYSIS\BASIC")  # 실제 BASIC 폴더로 수정
EP_SOURCES = [
    {"product": "KGVM", "axis": "y", "reference_scale": .5, "folder": EP_BASE_DIR / "edge_comparison_input" / "KGVM"},
    {"product": "KHVF", "axis": "y", "reference_scale": .2, "folder": EP_BASE_DIR / "edge_comparison_input" / "KHVF"},
    {"product": "PBVC", "axis": "x", "reference_scale": .4, "folder": EP_BASE_DIR / "edge_comparison_input" / "PBVC"},
]
EP_OUTPUT_DIR = EP_BASE_DIR / "output_edge_radius_sweep"
EP_INNER_RADIUS_UM = 144000.
EP_MAX_RADIUS_UM = 147000.
EP_STD_DDOF = 1
ER_FIT_RADII_MM = [146., 147.]
ER_SCALES = [i / 10 for i in range(1, 11)]
ER_SAVE_SCENARIO_POINTS = True  # edge가 있는 shot의 검산용 point만 배율/반경별 저장
ER_SAVE_COEFFICIENTS = True
ER_BASELINE_ATOL_UM = 1e-9
ER_SVD_RCOND = 1e-12

if not EP_SOURCES or len({str(s["product"]) for s in EP_SOURCES}) != len(EP_SOURCES):
    raise ValueError("제품명은 서로 달라야 합니다.")
if any(str(s.get("axis", "")).lower() not in {"x", "y"} for s in EP_SOURCES):
    raise ValueError("제품마다 axis를 x/y로 지정하세요.")
EP_PRODUCT_AXES = {str(s["product"]): [str(s["axis"]).lower()] for s in EP_SOURCES}
EP_AXES = sorted({a for axes in EP_PRODUCT_AXES.values() for a in axes})
EP_WAFER_KEYS = ["product", "LOT_ID", "Wafer"]
EP_SHOT_KEYS = EP_WAFER_KEYS + ["fcp_x", "fcp_y"]
if EP_STD_DDOF not in [0, 1] or not 0 < EP_INNER_RADIUS_UM < EP_MAX_RADIUS_UM:
    raise ValueError("표준편차 및 반경 설정을 확인하세요.")
if not ER_SCALES or any(not np.isfinite(s) or not 0 < s <= 1 for s in ER_SCALES):
    raise ValueError("ER_SCALES는 0 초과 1 이하 배율 목록이어야 합니다.")
ER_SCALES = sorted(set(float(s) for s in ER_SCALES))
if 1. not in ER_SCALES:
    raise ValueError("100% 조건은 반드시 포함하세요.")
ER_FIT_RADII_MM = sorted(set(float(r) for r in ER_FIT_RADII_MM))
if len(ER_FIT_RADII_MM) != 2 or any(not np.isfinite(r) or not EP_INNER_RADIUS_UM <= r*1000 <= EP_MAX_RADIUS_UM for r in ER_FIT_RADII_MM):
    raise ValueError("안쪽 평가 반경 이상, 147mm 이하의 회귀 반경 두 개를 지정하세요.")
for source in EP_SOURCES:
    if not any(np.isclose(source["reference_scale"], s) for s in ER_SCALES):
        raise ValueError(f"{source['product']}: reference_scale을 ER_SCALES 안의 값으로 지정하세요.")
for name in ["tables", "plots", "points", "coefficients", "diagnostics", "cases"]:
    (EP_OUTPUT_DIR / name).mkdir(parents=True, exist_ok=True)

def ep_locate_inputs(source):
    folder = Path(source["folder"])
    candidates = [folder / "27_OCO_FINAL_POINTS.parquet", folder / "stages" / "27_OCO_FINAL_POINTS.parquet",
                  folder / "27_OCO_FINAL_POINTS.xlsx", folder / "stages" / "27_OCO_FINAL_POINTS.xlsx"]
    existing = [p for p in candidates if p.is_file()]
    settings = folder / "extension_settings.json"
    if not existing or not settings.is_file():
        raise FileNotFoundError(f"{source['product']}: {folder}에 전체 27_OCO_FINAL_POINTS.parquet와 extension_settings.json을 두세요. stages 하위 파일 및 전체 Excel도 지원합니다. _sample 파일은 사용하지 않습니다.")
    return existing, settings

def ep_read_product(source):
    candidates, settings_path = ep_locate_inputs(source)
    errors = []
    for path in candidates:
        try:
            frame = pd.read_parquet(path) if path.suffix == ".parquet" else pd.read_excel(path)
            settings = json.loads(settings_path.read_text(encoding="utf-8"))
            return frame, settings, path, settings_path
        except (OSError, ValueError, ImportError) as exc:
            errors.append(f"{path.name}: {exc}")
    raise ValueError(f"{source['product']} 입력 읽기 실패: " + " | ".join(errors))

def ep_prepare_points(frame, product, axes=None):
    axes = EP_AXES if axes is None else axes
    required = ["LOT_ID", "Wafer", "fcp_x", "fcp_y", "coordinate_X", "coordinate_Y", "measurement_scale"] + [f"raw_minus_total_fit_{axis}" for axis in axes]
    missing = [col for col in required if col not in frame]
    if missing:
        raise ValueError(f"{product}: 27번 전체 point 자료에 필수 컬럼이 없습니다: {missing}")
    out = frame.copy().reset_index(drop=True)
    out["product"] = str(product)
    numeric = required[1:] + ["radius", "wf_x", "wf_y"] + [f"{axis.upper()}_reg_original" for axis in axes]
    for col in numeric:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if out.empty or out[EP_SHOT_KEYS].isna().any().any() or not np.isfinite(out[["Wafer", "fcp_x", "fcp_y"]].to_numpy(float)).all():
        raise ValueError(f"{product}: LOT/wafer/shot 식별자가 없거나 입력이 비어 있습니다.")
    if "photo_transn_seq" in out:
        photos = out.groupby(EP_WAFER_KEYS, observed=True, dropna=False).photo_transn_seq.nunique()
        if photos.gt(1).any():
            raise ValueError(f"{product}: 한 LOT/wafer에 여러 photo 이력이 있습니다. 실행별 자료를 섞지 말고 한 이력씩 분리하세요.")
    if "radius" not in out:
        out["radius"] = np.hypot(out.wf_x, out.wf_y) if {"wf_x", "wf_y"}.issubset(out.columns) else np.hypot(out.fcp_x + out.coordinate_X, out.fcp_y + out.coordinate_Y)
    elif out.radius.isna().any():
        fallback = np.hypot(out.wf_x, out.wf_y) if {"wf_x", "wf_y"}.issubset(out.columns) else np.hypot(out.fcp_x + out.coordinate_X, out.fcp_y + out.coordinate_Y)
        out["radius"] = out.radius.fillna(fallback)
    finite_scale = np.isfinite(out.measurement_scale)
    if not finite_scale.all() or out.measurement_scale.lt(0).any():
        raise ValueError(f"{product}: measurement_scale이 유한한 0 이상 값이어야 합니다.")
    scaled = out.measurement_scale.ne(1.)
    for axis in axes:
        col = f"{axis.upper()}_reg_original"
        needed = scaled & np.isfinite(out[f"raw_minus_total_fit_{axis}"])
        if needed.any() and (col not in out or not np.isfinite(out.loc[needed, col].to_numpy(float)).all()):
            raise ValueError(f"{product}: 배율 적용 전 값 복원에 필요한 {col}이 없습니다. 27번 전체 Parquet을 확인하세요.")
        restore = pd.Series(0., index=out.index)
        if needed.any():
            restore.loc[needed] = (1. - out.loc[needed, "measurement_scale"]) * out.loc[needed, col]
        # 기적용 MRC/PSM decorrection과 고정 K-MRC는 유지. 측정값 배율만 100%로 복원.
        out[f"input100_{axis}"] = out[f"raw_minus_total_fit_{axis}"] + restore
    in_range = np.isfinite(out.radius) & out.radius.ge(0) & out.radius.le(EP_MAX_RADIUS_UM)
    audit = dict(product=product, axis="/".join(axes), n_input=len(out), n_radius_unknown=int((~np.isfinite(out.radius)).sum()),
                 n_outside_radius=int((np.isfinite(out.radius) & ~in_range).sum()), n_scaled=int(scaled.sum()))
    out = out.loc[in_range].copy().reset_index(drop=True)
    if out.empty:
        raise ValueError(f"{product}: 분석 반경 안의 측정점이 없습니다.")
    out["region"] = np.where(out.radius.le(EP_INNER_RADIUS_UM), "INNER", "EDGE")
    out["_fit_valid"] = np.isfinite(out[["coordinate_X", "coordinate_Y"] + [f"input100_{axis}" for axis in axes]].to_numpy(float)).all(axis=1)
    out["_has_edge_shot"] = out.region.eq("EDGE").groupby([out[k] for k in EP_SHOT_KEYS], observed=True, dropna=False).transform("any")
    audit.update(n_used=len(out), n_inner=int(out.region.eq("INNER").sum()), n_edge=int(out.region.eq("EDGE").sum()), n_fit_valid=int(out._fit_valid.sum()))
    return out, audit

print("기준 폴더:", EP_BASE_DIR)
er_inventory = []
for source in EP_SOURCES:
    files, settings = ep_locate_inputs(source)
    er_inventory.append(dict(product=source["product"], axis=EP_PRODUCT_AXES[str(source["product"])][0], points=" | ".join(str(p) for p in files), settings=str(settings)))
display(pd.DataFrame(er_inventory))
print(f"제품별 {len(ER_FIT_RADII_MM)*len(ER_SCALES)}조건. K-MRC 고정 / 회귀 point만 제한 / 평가는 원래 값으로 동일 영역 사용.")


## 01. 통계 및 저장 설정 복원 함수


In [ ]:
# %% 01. 통계·제품별 CPE 설정 복원 함수
EP_STAT_COLUMNS = ["n_points", "n_valid", "mean_nm", "abs_mean_nm", "sigma_nm", "three_sigma_nm", "M3S_nm", "RMSE_nm", "p95_abs_nm"]


def ep_group_stats(frame, keys, value_col, skeleton=None):
    result = frame[keys].drop_duplicates().copy() if skeleton is None else skeleton[keys].drop_duplicates().copy()
    if frame.empty:
        for col in EP_STAT_COLUMNS:
            result[col] = 0 if col in ["n_points", "n_valid"] else np.nan
        return result
    work = frame[keys].copy()
    values = pd.to_numeric(frame[value_col], errors="coerce") * 1000
    work["_v"] = values.where(np.isfinite(values))
    work["_v2"], work["_abs"] = work._v ** 2, work._v.abs()
    groups = work.groupby(keys, observed=True, dropna=False)
    stats = groups._v.agg(n_points="size", n_valid="count", mean_nm="mean")
    stats["sigma_nm"] = groups._v.std(ddof=EP_STD_DDOF)
    stats["RMSE_nm"] = np.sqrt(groups._v2.mean())
    stats["p95_abs_nm"] = groups._abs.quantile(.95)
    result = result.merge(stats.reset_index(), on=keys, how="left", validate="1:1")
    result[["n_points", "n_valid"]] = result[["n_points", "n_valid"]].fillna(0).astype(int)
    result["abs_mean_nm"] = result.mean_nm.abs()
    result["three_sigma_nm"] = 3 * result.sigma_nm
    result["M3S_nm"] = result.abs_mean_nm + result.three_sigma_nm
    return result


def ep_input_stats(points, keys, axes=None):
    axes = EP_AXES if axes is None else axes
    tables = []
    for axis in axes:
        for region in ["INNER", "EDGE"]:
            table = ep_group_stats(points.loc[points.region.eq(region)], keys, f"input100_{axis}", skeleton=points)
            table["axis"], table["region"] = axis, region
            tables.append(table)
    return pd.concat(tables, ignore_index=True)


def ep_region_comparison(stats, keys):
    columns = keys + ["axis"]
    inner = stats.loc[stats.region.eq("INNER")].drop(columns="region")
    edge = stats.loc[stats.region.eq("EDGE")].drop(columns="region")
    result = inner.merge(edge, on=columns, suffixes=("_inner", "_edge"), validate="1:1")
    result["delta_M3S_edge_minus_inner_nm"] = result.M3S_nm_edge - result.M3S_nm_inner
    result["ratio_3sigma_edge_to_inner"] = result.three_sigma_nm_edge / result.three_sigma_nm_inner.replace(0, np.nan)
    return result


def ep_restore_settings(settings):
    required = ["cpe_options", "use_virtual_chip_points", "virtual_wafer_radius_um", "span_tolerance", "minimum_ridge", "add_ridge", "max_gopt_return", "max_span_iter", "cd_tol", "ridge_valves", "active_span_limits"]
    missing = [key for key in required if key not in settings]
    if missing:
        raise ValueError(f"28번 실행 설정에 필수 항목이 없습니다: {missing}")
    if settings.get("kmrc_definition") != "OCO_E1_RAW_MINUS_ADI_RAW":
        raise ValueError("현재 K-MRC(E1 raw−ADI raw) 방식의 27·28번 저장 자료가 필요합니다. 이전 E2 자체 regression 자료와 섞을 수 없습니다.")
    options = copy.deepcopy(settings["cpe_options"])
    globals_map = {"use_virtual_chip_points": "EP_USE_VIRTUAL_CHIP_POINTS", "virtual_wafer_radius_um": "EP_VIRTUAL_WAFER_RADIUS", "span_tolerance": "EP_SPAN_TOLERANCE",
                   "minimum_ridge": "EP_MINIMUM_RIDGE", "add_ridge": "EP_ADD_RIDGE", "max_gopt_return": "EP_MAX_GOPT_RETURN", "max_span_iter": "EP_MAX_SPAN_ITER", "cd_tol": "EP_CD_TOL"}
    for key, variable in globals_map.items():
        globals()[variable] = settings[key]
    globals()["EP_RIDGE_VALVE_HYPER"] = copy.deepcopy(settings["ridge_valves"])
    span_variables = {"OVO3_38P_HARDLIMIT": "EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT", "OVO3_38P_F2F": "EP_RK_SPAN_LIMITS_OVO3_38P_F2F"}
    if options["span_mode"] not in span_variables:
        raise ValueError(f"지원하지 않는 span_mode: {options['span_mode']}")
    globals()[span_variables[options["span_mode"]]] = copy.deepcopy(settings["active_span_limits"])
    return options


## 02. 기존 CPE38 모델·virtual grid


In [ ]:
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

EP_SPAN_MODE = "OVO3_38P_F2F"
EP_RIDGE_REGRESSION = True
EP_G_OPT_SPEC = 1.0
EP_USE_VIRTUAL_CHIP_POINTS = True   # VBA virtual grid: chip별 4 corner + center point
EP_G_OPT_TOLERANCE = 0.001          # VBA: G_opt_tolerlance = 0.001
EP_SPAN_TOLERANCE = 0.01            # VBA: span_tolerlance = 0.01
EP_MINIMUM_RIDGE = 0.001            # VBA: minimum_Ridge = 0.001
EP_ADD_RIDGE = 10.0                 # VBA: add_ridge = 10
EP_MAX_GOPT_RETURN = 50             # VBA: max_prevent_fallback_retrun = 50
EP_MAX_SPAN_ITER = 5000             # VBA: max_iter = 5000
EP_CD_TOL = 1e-6                    # VBA: tol = 0.000001
EP_CPE_EVAL_GRID_N = 10
EP_MIN_POINTS_GUIDE = 10
EP_G_OPTIMAL_TYPE = 1               # VBA information 기준: GOT1
EP_VIRTUAL_WAFER_RADIUS = 144000.0  # VBA virtual point keep 조건: R <= 144000               # VBA option D59: 1,2,3,4. 4 = inner OT2 / partial OT1
EP_USE_SPAN_BOX_CONSTRAINT = True   # VBA Balloon projected CD/PGD stage

def ep_cpe_38para(rx: Sequence[float], ry: Sequence[float]) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    X_dx = np.vstack([
        np.ones(len(rx)),
        rx / 1e6, -ry / 1e6,
        (rx ** 2) / 1e9, (rx * ry) / 1e9, (ry ** 2) / 1e9,
        (rx ** 3) / 1e12, (rx ** 2 * ry) / 1e12, (rx * ry ** 2) / 1e12, (ry ** 3) / 1e12,
        (rx ** 3 * ry) / 1e19, (rx ** 2 * ry ** 2) / 1e19, (rx * ry ** 3) / 1e19, (ry ** 4) / 1e19,
        (rx ** 3 * ry ** 2) / 1e23, (rx ** 2 * ry ** 3) / 1e23, (rx * ry ** 4) / 1e23, (ry ** 5) / 1e23,
        (rx ** 3 * ry ** 3) / 1e27, (rx ** 2 * ry ** 4) / 1e27,
        (rx ** 3 * ry ** 4) / 1e31,
    ]).T
    X_dy = np.vstack([
        np.ones(len(ry)),
        ry / 1e6, rx / 1e6,
        (ry ** 2) / 1e9, (ry * rx) / 1e9, (rx ** 2) / 1e9,
        (ry ** 3) / 1e12, (ry ** 2 * rx) / 1e12, (ry * rx ** 2) / 1e12,
        (ry ** 4) / 1e19, (ry ** 3 * rx) / 1e19, (ry ** 2 * rx ** 2) / 1e19,
        (ry ** 5) / 1e23, (ry ** 4 * rx) / 1e23, (ry ** 3 * rx ** 2) / 1e23,
        (ry ** 5 * rx) / 1e27, (ry ** 4 * rx ** 2) / 1e27,
    ]).T
    return X_dx, X_dy
EP_CPE38_KEYS_DX = [
    "RK1", "RK3", "RK5",
    "RK7", "RK9", "RK11",
    "RK13", "RK15", "RK17", "RK19",
    "RK23", "RK25", "RK27", "RK29",
    "RK35", "RK37", "RK39", "RK41",
    "RK49", "RK51",
    "RK65",
]
EP_CPE38_KEYS_DY = [
    "RK2", "RK4", "RK6",
    "RK8", "RK10", "RK12",
    "RK14", "RK16", "RK18",
    "RK22", "RK24", "RK26",
    "RK32", "RK34", "RK36",
    "RK46", "RK48",
]
EP_RK_SPAN_LIMITS_OVO3_38P_F2F = {
    'RK1': {'min': -0.1, 'max': 0.1},
    'RK2': {'min': -0.1, 'max': 0.1},
    'RK3': {'min': -2.0, 'max': 2.0},
    'RK4': {'min': -10.0, 'max': 10.0},
    'RK5': {'min': -5.0, 'max': 5.0},
    'RK6': {'min': -5.0, 'max': 5.0},
    'RK7': {'min': -0.1, 'max': 0.1},
    'RK8': {'min': -0.15, 'max': 0.15},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -0.5, 'max': 0.5},
    'RK11': {'min': -0.5, 'max': 0.5},
    'RK12': {'min': -0.03, 'max': 0.03},
    'RK13': {'min': -0.003, 'max': 0.003},
    'RK14': {'min': -0.015, 'max': 0.015},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -0.05, 'max': 0.05},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -0.023, 'max': 0.023},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}
EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT = {
    'RK1': {'min': -100.0, 'max': 100.0},
    'RK2': {'min': -100.0, 'max': 100.0},
    'RK3': {'min': -1000.0, 'max': 1000.0},
    'RK4': {'min': -1000.0, 'max': 1000.0},
    'RK5': {'min': -1000.0, 'max': 1000.0},
    'RK6': {'min': -1000.0, 'max': 1000.0},
    'RK7': {'min': -0.25, 'max': 0.25},
    'RK8': {'min': -10.0, 'max': 10.0},
    'RK9': {'min': -0.06, 'max': 0.06},
    'RK10': {'min': -10.0, 'max': 10.0},
    'RK11': {'min': -10.0, 'max': 10.0},
    'RK12': {'min': -0.25, 'max': 0.25},
    'RK13': {'min': -0.025, 'max': 0.025},
    'RK14': {'min': -1.0, 'max': 1.0},
    'RK15': {'min': -0.002, 'max': 0.002},
    'RK16': {'min': -1.0, 'max': 1.0},
    'RK17': {'min': -0.005, 'max': 0.005},
    'RK18': {'min': -0.002, 'max': 0.002},
    'RK19': {'min': -1.0, 'max': 1.0},
    'RK22': {'min': -7.0, 'max': 7.0},
    'RK23': {'min': -2.5, 'max': 2.5},
    'RK24': {'min': -18.0, 'max': 18.0},
    'RK25': {'min': -8.0, 'max': 8.0},
    'RK26': {'min': -4.0, 'max': 4.0},
    'RK27': {'min': -7.0, 'max': 7.0},
    'RK29': {'min': -7.0, 'max': 7.0},
    'RK32': {'min': -2.5, 'max': 2.5},
    'RK34': {'min': -5.0, 'max': 5.0},
    'RK35': {'min': -2.0, 'max': 2.0},
    'RK36': {'min': -1.2, 'max': 1.2},
    'RK37': {'min': -2.5, 'max': 2.5},
    'RK39': {'min': -4.0, 'max': 4.0},
    'RK41': {'min': -2.5, 'max': 2.5},
    'RK46': {'min': -1.2, 'max': 1.2},
    'RK48': {'min': -0.6, 'max': 0.6},
    'RK49': {'min': -0.8, 'max': 0.8},
    'RK51': {'min': -1.2, 'max': 1.2},
    'RK65': {'min': -0.7, 'max': 0.7},
}

def ep_get_active_span_limits(span_mode: str = "OVO3_38P_F2F") -> Dict[str, Dict[str, float]]:
    if span_mode == "OVO3_38P_HARDLIMIT":
        return EP_RK_SPAN_LIMITS_OVO3_38P_HARDLIMIT
    return EP_RK_SPAN_LIMITS_OVO3_38P_F2F
EP_RIDGE_VALVE_HYPER = {
    "RK1": 0.0, "RK2": 0.0,
    "RK3": 0.01, "RK4": 0.01, "RK5": 0.01, "RK6": 0.01,
    "RK7": 1.0, "RK8": 0.1, "RK9": 1.0, "RK10": 0.1,
    "RK11": 0.1, "RK12": 1.0,
    "RK13": 1.0, "RK14": 0.1, "RK15": 1.0, "RK16": 0.1,
    "RK17": 1.0, "RK18": 1.0, "RK19": 0.1, "RK20": 1.0,
}

def ep__to_numeric(df: pd.DataFrame, cols: Iterable[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def ep_choose_base_lambda_scale(n_points: int) -> float:
    """VBA의 point가 적어도 skip하지 않는 방식을 Python식으로 반영."""
    if n_points >= 10:
        return 1.0
    if n_points >= 7:
        return 1e2
    if n_points >= 5:
        return 1e4
    if n_points >= 3:
        return 1e6
    return 1e8

def ep_get_ridge_valves(coeff_names: Sequence[str]) -> np.ndarray:
    return np.array([EP_RIDGE_VALVE_HYPER.get(name, 1.0) for name in coeff_names], dtype=float)

def ep_make_lambda_vector(base_lambda: float, coeff_names: Sequence[str]) -> np.ndarray:
    valves = ep_get_ridge_valves(coeff_names)
    return float(base_lambda) * valves

def ep_build_virtual_chip_points(
    step_pitch_x: float,
    step_pitch_y: float,
    chip_x_num: int,
    chip_y_num: int,
    round_decimals: int = 6,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA virtual grid 재현:
    chip별 4 corner + center point를 shot-local coordinate로 생성하고,
    chip 경계에서 겹치는 corner는 dedup한다.
    예: chip_x_num=2, chip_y_num=1
      - 각 chip 5 point => 10 raw points
      - 가운데 공유 corner 2개 중복 제거
      - 최종 8 virtual points
    coordinate_X/Y가 shot center 기준이라고 보고,
    전체 shot 영역은 [-STEP_PITCH_X/2, +STEP_PITCH_X/2],
                  [-STEP_PITCH_Y/2, +STEP_PITCH_Y/2] 로 둔다.
    """
    chip_x_num = int(chip_x_num) if pd.notna(chip_x_num) and int(chip_x_num) > 0 else 1
    chip_y_num = int(chip_y_num) if pd.notna(chip_y_num) and int(chip_y_num) > 0 else 1
    sx = float(step_pitch_x)
    sy = float(step_pitch_y)
    x_edges = np.linspace(-sx / 2.0, sx / 2.0, chip_x_num + 1)
    y_edges = np.linspace(-sy / 2.0, sy / 2.0, chip_y_num + 1)
    pts = []
    for ix in range(chip_x_num):
        for iy in range(chip_y_num):
            x0, x1 = x_edges[ix], x_edges[ix + 1]
            y0, y1 = y_edges[iy], y_edges[iy + 1]
            xc = (x0 + x1) / 2.0
            yc = (y0 + y1) / 2.0
            pts.extend([
                (x0, y0), (x0, y1), (x1, y0), (x1, y1),
                (xc, yc),
            ])
    pts_unique = sorted({(round(x, round_decimals), round(y, round_decimals)) for x, y in pts})
    rx = np.array([p[0] for p in pts_unique], dtype=float)
    ry = np.array([p[1] for p in pts_unique], dtype=float)
    return rx, ry

def ep_filter_virtual_points_by_wafer_radius(
    rx_v: Sequence[float],
    ry_v: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = EP_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    VBA edge-shot virtual point filtering.
    virtual point의 wafer 좌표 R이 wafer_radius 이하인 point만 사용한다.
    R = sqrt((fcp_x + rx_v)^2 + (fcp_y + ry_v)^2)
    keep if R <= 144000
    """
    rx_v = np.asarray(rx_v, dtype=float)
    ry_v = np.asarray(ry_v, dtype=float)
    wx = float(fcp_x) + rx_v
    wy = float(fcp_y) + ry_v
    r = np.sqrt(wx * wx + wy * wy)
    keep = r <= float(wafer_radius)
    return rx_v[keep], ry_v[keep], r[keep]

def ep_get_virtual_points_for_group(
    grp: pd.DataFrame,
    fallback_rx: Sequence[float],
    fallback_ry: Sequence[float],
    fcp_x: float,
    fcp_y: float,
    wafer_radius: float = EP_VIRTUAL_WAFER_RADIUS,
) -> Tuple[np.ndarray, np.ndarray, str, int, int]:
    """
    1) full virtual point 생성: chip별 4 corner + center
    2) edge shot 처리: wafer 좌표 R <= wafer_radius point만 keep
    반환:
      rx_v, ry_v, virtual_source, n_virtual_full, n_virtual_kept
    """
    required = ["STEP_PITCH_X", "STEP_PITCH_Y", "CHIP_X_NUM", "CHIP_Y_NUM"]
    if all(c in grp.columns for c in required):
        vals = grp[required].dropna()
        if len(vals) > 0:
            row = vals.iloc[0]
            rx_full, ry_full = ep_build_virtual_chip_points(
                step_pitch_x=row["STEP_PITCH_X"],
                step_pitch_y=row["STEP_PITCH_Y"],
                chip_x_num=row["CHIP_X_NUM"],
                chip_y_num=row["CHIP_Y_NUM"],
            )
            n_full = len(rx_full)
            rx_keep, ry_keep, _ = ep_filter_virtual_points_by_wafer_radius(
                rx_full,
                ry_full,
                fcp_x=fcp_x,
                fcp_y=fcp_y,
                wafer_radius=wafer_radius,
            )
            if len(rx_keep) > 0:
                src = "chip_corner_center_wafer_clipped"
                return rx_keep, ry_keep, src, n_full, len(rx_keep)
            return rx_full, ry_full, "chip_corner_center_no_clip_points_fallback", n_full, n_full
    rx_v, ry_v = ep_build_eval_grid_from_data(fallback_rx, fallback_ry, grid_n=EP_CPE_EVAL_GRID_N)
    return rx_v, ry_v, "measured_range_grid_fallback", len(rx_v), len(rx_v)


## 03. 기존 ridge/G-opt/span solver


In [ ]:
def ep_stable_inverse_or_pinv(A: np.ndarray) -> np.ndarray:
    try:
        return np.linalg.inv(A)
    except np.linalg.LinAlgError:
        return np.linalg.pinv(A)

def ep_ridge_predict(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X.T @ X + np.diag(lambdas)
    b = X.T @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(A) @ b
    return beta, X @ beta

def ep_build_eval_grid_from_data(rx: Sequence[float], ry: Sequence[float], grid_n: int = 10) -> Tuple[np.ndarray, np.ndarray]:
    rx = np.asarray(rx, dtype=float)
    ry = np.asarray(ry, dtype=float)
    rx_min, rx_max = np.nanmin(rx), np.nanmax(rx)
    ry_min, ry_max = np.nanmin(ry), np.nanmax(ry)
    if not np.isfinite(rx_min) or not np.isfinite(rx_max):
        rx_min, rx_max = -1.0, 1.0
    if not np.isfinite(ry_min) or not np.isfinite(ry_max):
        ry_min, ry_max = -1.0, 1.0
    if rx_min == rx_max:
        rx_min -= 1.0
        rx_max += 1.0
    if ry_min == ry_max:
        ry_min -= 1.0
        ry_max += 1.0
    gx = np.linspace(rx_min, rx_max, grid_n)
    gy = np.linspace(ry_min, ry_max, grid_n)
    RX, RY = np.meshgrid(gx, gy)
    return RX.ravel(), RY.ravel()

def ep_gopt_value(
    X_meas: np.ndarray,
    X_eval: np.ndarray,
    lambdas: np.ndarray,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
) -> float:
    """
    VBA G_optimal_type 반영.
    - Type1: H H.T diag의 sqrt(max)
    - Type2: H matrix element의 signed max의 sqrt. 음수이면 999.0 반환.
    - Type3: X_eval inv(...) X_eval.T diag의 sqrt(max)
    - Type4: inner=Type2, partial=Type1
    """
    X_meas = np.asarray(X_meas, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    A = X_meas.T @ X_meas + np.diag(lambdas)
    invA = ep_stable_inverse_or_pinv(A)
    effective_type = gopt_type
    if gopt_type == 4:
        effective_type = 2 if inner_or_partial == "inner" else 1
    if effective_type == 1:
        H = X_eval @ invA @ X_meas.T
        diag_val = np.sum(H * H, axis=1)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    if effective_type == 2:
        H = X_eval @ invA @ X_meas.T
        max_val = float(np.nanmax(H))
        if max_val < 0:
            return 999.0
        return float(np.sqrt(max_val))
    if effective_type == 3:
        M = X_eval @ invA @ X_eval.T
        diag_val = np.diag(M)
        max_val = np.nanmax(diag_val)
        return float(np.sqrt(max(max_val, 0.0)))
    raise ValueError(f"Unsupported gopt_type: {gopt_type}")

def ep_calc_contribution_spans(X_eval: np.ndarray, beta: np.ndarray, coeff_names: Sequence[str]) -> Dict[str, float]:
    spans = {}
    for i, name in enumerate(coeff_names):
        contrib = X_eval[:, i] * beta[i]
        spans[name] = float(np.nanmax(contrib) - np.nanmin(contrib))
    return spans

def ep_find_span_violations(
    spans: Dict[str, float],
    span_limits: Dict[str, Dict[str, float]],
    tolerance: float = 0.01,
) -> List[Tuple[str, float, float, float]]:
    violations = []
    for name, span in spans.items():
        if name not in span_limits:
            continue
        lo = span_limits[name]["min"] - tolerance
        hi = span_limits[name]["max"] + tolerance
        if span < lo or span > hi:
            violations.append((name, span, span_limits[name]["min"], span_limits[name]["max"]))
    return violations

def ep_build_beta_bounds_from_span(
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
) -> Tuple[np.ndarray, np.ndarray]:
    """
    VBA의 min_Span_X/Y, max_Span_X/Y에 대응하는 box constraint 근사.
    contribution span = range(X_j * beta_j)가 span_limit 안에 들어오도록
    beta_j bound를 계산한다.
    """
    n = len(coeff_names)
    lo = np.full(n, -np.inf, dtype=float)
    hi = np.full(n, np.inf, dtype=float)
    for j, name in enumerate(coeff_names):
        if name not in span_limits:
            continue
        lim = span_limits[name]
        span_abs = max(abs(float(lim["min"])), abs(float(lim["max"])))
        xj_range = float(np.nanmax(X_eval[:, j]) - np.nanmin(X_eval[:, j]))
        if xj_range <= 0 or not np.isfinite(xj_range):
            continue
        b = span_abs / xj_range
        lo[j] = -b
        hi[j] = b
    return lo, hi

def ep_projected_coordinate_descent_ridge(
    X: np.ndarray,
    y: np.ndarray,
    lambdas: np.ndarray,
    beta_init: Optional[np.ndarray] = None,
    lower: Optional[np.ndarray] = None,
    upper: Optional[np.ndarray] = None,
    tol: float = 1e-6,
    max_iter: int = 5000,
) -> Tuple[np.ndarray, np.ndarray, int, float]:
    """
    VBA ProjectedCoordinateDescentElasticNet(l1=0) 근사.
    solve min ||y-Xb||^2 + sum(lambda_j*b_j^2), s.t. lower <= b <= upper
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    lambdas = np.asarray(lambdas, dtype=float)
    n_param = X.shape[1]
    if beta_init is None:
        beta, _ = ep_ridge_predict(X, y, lambdas)
    else:
        beta = np.asarray(beta_init, dtype=float).copy()
    if lower is None:
        lower = np.full(n_param, -np.inf)
    if upper is None:
        upper = np.full(n_param, np.inf)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    beta = np.clip(beta, lower, upper)
    residual = y - X @ beta
    col_norm2 = np.sum(X * X, axis=0)
    last_delta = np.inf
    for it in range(1, max_iter + 1):
        max_delta = 0.0
        for j in range(n_param):
            denom = col_norm2[j] + lambdas[j]
            if denom <= 0 or not np.isfinite(denom):
                continue
            residual += X[:, j] * beta[j]
            rho = float(X[:, j] @ residual)
            new_beta = rho / denom
            new_beta = min(max(new_beta, lower[j]), upper[j])
            delta = abs(new_beta - beta[j])
            beta[j] = new_beta
            residual -= X[:, j] * beta[j]
            if delta > max_delta:
                max_delta = delta
        last_delta = max_delta
        if max_delta < tol:
            break
    return beta, X @ beta, it, float(last_delta)

@dataclass
class EP_RidgeFitResult:
    beta: np.ndarray
    pred: np.ndarray
    lambdas: np.ndarray
    gopt: float
    spans: Dict[str, float]
    violations: List[Tuple[str, float, float, float]]
    gopt_iter: int
    span_iter: int
    method: str

def ep_solve_vba_style_ridge_one_axis(
    X_fit: np.ndarray,
    y: np.ndarray,
    X_eval: np.ndarray,
    coeff_names: Sequence[str],
    span_limits: Dict[str, Dict[str, float]],
    ridge_regression: bool = True,
    gopt_spec: float = 1.0,
    gopt_tolerance: float = 0.001,
    minimum_ridge: float = 0.001,
    add_ridge: float = 10.0,
    max_gopt_return: int = 50,
    gopt_type: int = 4,
    inner_or_partial: str = "partial",
    use_span_box_constraint: bool = True,
    span_tolerance: float = 0.01,
    cd_tol: float = 1e-6,
    max_span_iter: int = 5000,
    base_lambda_scale: float = 1.0,
    debug: bool = False,
) -> EP_RidgeFitResult:
    """
    VBA flow에 맞춘 한 방향(X or Y) fitting.
    - 초기 ridge는 0에 가까운 값이 아니라 VBA minimum_Ridge 기준 사용
    - Gopt가 spec 이상이면 ridge를 빠르게 키운 뒤 이분탐색으로 spec 근처를 찾음
    - span over가 남으면 projected coordinate descent로 coefficient를 box constraint 안에 넣음
    """
    X_fit = np.asarray(X_fit, dtype=float)
    X_eval = np.asarray(X_eval, dtype=float)
    y = np.asarray(y, dtype=float)
    n_param = X_fit.shape[1]
    if n_param != len(coeff_names):
        raise ValueError(f"n_param({n_param}) != len(coeff_names)({len(coeff_names)})")
    initial_lambda = 0.0 if not ridge_regression else 0.0
    lambdas = ep_make_lambda_vector(initial_lambda, coeff_names)
    beta, pred = ep_ridge_predict(X_fit, y, lambdas)
    g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
    gopt_iter = 0
    method = "ols"
    if ridge_regression and (not np.isfinite(g) or g >= gopt_spec or g == 0):
        low_lambda = 0.0
        high_lambda = max(minimum_ridge * base_lambda_scale, minimum_ridge)
        high_lambda = max(high_lambda, minimum_ridge)
        high_lambdas = ep_make_lambda_vector(high_lambda, coeff_names)
        high_g = ep_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        while (not np.isfinite(high_g) or high_g >= gopt_spec) and gopt_iter < max_gopt_return:
            low_lambda = high_lambda
            high_lambda = high_lambda * add_ridge
            high_lambdas = ep_make_lambda_vector(high_lambda, coeff_names)
            high_g = ep_gopt_value(X_fit, X_eval, high_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
            gopt_iter += 1
            if debug:
                print(f"fast ridge: lambda={high_lambda:.6g}, gopt={high_g:.6g}")
        target_low = max(0.0, gopt_spec - gopt_tolerance)
        target_high = gopt_spec
        best_lambda = high_lambda
        best_g = high_g
        if np.isfinite(high_g) and high_g < gopt_spec:
            lo = low_lambda
            hi = high_lambda
            for _ in range(max_gopt_return):
                mid = (lo + hi) / 2.0
                mid_lambdas = ep_make_lambda_vector(mid, coeff_names)
                mid_g = ep_gopt_value(X_fit, X_eval, mid_lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
                gopt_iter += 1
                if debug:
                    print(f"slow ridge: lambda={mid:.6g}, gopt={mid_g:.6g}")
                if target_low <= mid_g < target_high:
                    best_lambda = mid
                    best_g = mid_g
                    break
                if mid_g >= gopt_spec or not np.isfinite(mid_g):
                    lo = mid
                else:
                    hi = mid
                    best_lambda = mid
                    best_g = mid_g
                if abs(hi - lo) < max(minimum_ridge * 1e-3, 1e-12):
                    break
        lambdas = ep_make_lambda_vector(best_lambda, coeff_names)
        beta, pred = ep_ridge_predict(X_fit, y, lambdas)
        g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt"
    spans = ep_calc_contribution_spans(X_eval, beta, coeff_names)
    violations = ep_find_span_violations(spans, span_limits, tolerance=span_tolerance)
    span_iter = 0
    if ridge_regression and use_span_box_constraint and len(violations) > 0:
        lower, upper = ep_build_beta_bounds_from_span(X_eval, coeff_names, span_limits)
        beta_cd, pred_cd, span_iter, _ = ep_projected_coordinate_descent_ridge(
            X_fit,
            y,
            lambdas,
            beta_init=beta,
            lower=lower,
            upper=upper,
            tol=cd_tol,
            max_iter=max_span_iter,
        )
        beta = beta_cd
        pred = pred_cd
        spans = ep_calc_contribution_spans(X_eval, beta, coeff_names)
        violations = ep_find_span_violations(spans, span_limits, tolerance=span_tolerance)
        g = ep_gopt_value(X_fit, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)
        method = "ridge_gopt_box_cd"
    return EP_RidgeFitResult(
        beta=beta,
        pred=pred,
        lambdas=lambdas,
        gopt=float(g),
        spans=spans,
        violations=violations,
        gopt_iter=int(gopt_iter),
        span_iter=int(span_iter),
        method=method,
    )

def ep_determine_inner_or_partial(n_valid: int, n_virtual: int) -> str:
    """VBA: virtual_points = Vgrid_XY면 inner, 아니면 partial."""
    return "inner" if n_valid >= n_virtual else "partial"


## 04. Shot 회귀·캐시·평가·재현 검사


In [ ]:
er_native_gopt = getattr(ep_gopt_value, "_er_uncached_native", ep_gopt_value)


@lru_cache(maxsize=2048)
def er_cached_geometry_gopt(x_shape, x_bytes, eval_shape, eval_bytes, ridge_bytes, gopt_type, inner_or_partial):
    X = np.frombuffer(x_bytes, dtype=np.float64).reshape(x_shape)
    X_eval = np.frombuffer(eval_bytes, dtype=np.float64).reshape(eval_shape)
    lambdas = np.frombuffer(ridge_bytes, dtype=np.float64)
    return er_native_gopt(X, X_eval, lambdas, gopt_type=gopt_type, inner_or_partial=inner_or_partial)


def er_gopt_value(X_meas, X_eval, lambdas, gopt_type=4, inner_or_partial="partial"):
    X, V, ridge = np.ascontiguousarray(X_meas, dtype=np.float64), np.ascontiguousarray(X_eval, dtype=np.float64), np.ascontiguousarray(lambdas, dtype=np.float64)
    return er_cached_geometry_gopt(X.shape, X.tobytes(), V.shape, V.tobytes(), ridge.tobytes(), gopt_type, inner_or_partial)


er_gopt_value._er_uncached_native = er_native_gopt
ep_gopt_value = er_gopt_value  # 동일 좌표·lambda의 G-opt만 캐시. 계수·span은 측정값별 재계산


def er_prepare_product(raw, product, axis):
    points, audit = ep_prepare_points(raw, product, axes=[axis])
    original = f"{axis.upper()}_reg_original"
    needed = points._fit_valid & points.region.eq("EDGE")
    if needed.any() and (original not in points or not np.isfinite(pd.to_numeric(points.loc[needed, original], errors="coerce")).all()):
        raise ValueError(f"{product}: edge 배율 변경에 필요한 {original}이 없습니다.")
    inner = points.region.eq("INNER") & points._fit_valid
    edge = points.region.eq("EDGE") & points._fit_valid
    groups = [points[k] for k in EP_SHOT_KEYS]
    points["_mixed_shot"] = inner.groupby(groups, observed=True, dropna=False).transform("any") & edge.groupby(groups, observed=True, dropna=False).transform("any")
    return points, audit


def er_matrix_geometry(X):
    if not len(X):
        return dict(rank=0, scaled_condition=np.inf)
    norms = np.linalg.norm(X, axis=0)
    singular = np.linalg.svd(X/np.where(norms > 0, norms, 1.), compute_uv=False)
    rank = int((singular > singular[0]*ER_SVD_RCOND).sum()) if len(singular) else 0
    return dict(rank=rank, scaled_condition=float(singular[0]/singular[-1]) if rank == X.shape[1] else np.inf)


def er_build_contexts(points, axis, options):
    contexts = []
    matrix_index = 0 if axis == "x" else 1
    names = EP_CPE38_KEYS_DX if axis == "x" else EP_CPE38_KEYS_DY
    for key, shot in points.groupby(EP_SHOT_KEYS, observed=True, dropna=False):
        work = shot.loc[shot._fit_valid]
        identity = dict(zip(EP_SHOT_KEYS, key))
        identity.update({col: shot[col].iloc[0] if col in shot else np.nan for col in ["DieX", "DieY", "photo_transn_seq"]})
        context = dict(identity=identity, indices=work.index.to_numpy(), shot_indices=shot.index.to_numpy(), affected=bool(shot._has_edge_shot.any()),
                       radius=work.radius.to_numpy(float), input100=work[f"input100_{axis}"].to_numpy(float), baseline_input=work[f"raw_minus_total_fit_{axis}"].to_numpy(float),
                       X=ep_cpe_38para(work.coordinate_X, work.coordinate_Y)[matrix_index], names=names, axis=axis, options=options, cache={}, masks={})
        original = pd.Series(0., index=work.index)
        outer = work.region.eq("EDGE")
        if outer.any():
            original.loc[outer] = pd.to_numeric(work.loc[outer, f"{axis.upper()}_reg_original"], errors="coerce")
        context["edge_original"] = original.to_numpy(float)
        if len(work):
            if EP_USE_VIRTUAL_CHIP_POINTS:
                vx, vy, source, n_full, n_virtual = ep_get_virtual_points_for_group(work, work.coordinate_X, work.coordinate_Y, fcp_x=key[-2], fcp_y=key[-1], wafer_radius=EP_VIRTUAL_WAFER_RADIUS)
            else:
                vx, vy = ep_build_eval_grid_from_data(work.coordinate_X, work.coordinate_Y, grid_n=options["eval_grid_n"])
                source, n_full, n_virtual = "measured_range_grid", len(vx), len(vx)
            context.update(Xv=ep_cpe_38para(vx, vy)[matrix_index], virtual_source=source, n_virtual=n_virtual, n_virtual_full=n_full)
        for radius in ER_FIT_RADII_MM:
            mask = context["radius"] <= radius*1000
            context["masks"][radius] = (mask, er_matrix_geometry(context["X"][mask]))
        contexts.append(context)
    return contexts


def er_solve_context(context, fit_radius_mm, scale=None):
    baseline = scale is None
    mask = np.ones(len(context["indices"]), dtype=bool) if baseline else context["masks"][fit_radius_mm][0]
    # 같은 shot에서 두 반경의 학습 point가 같으면 배율별 해를 재사용한다.
    cache_key = ("baseline",) if baseline else (mask.tobytes(), float(scale))
    if cache_key in context["cache"]:
        return context["cache"][cache_key]
    if not baseline and mask.all() and not np.any(context["edge_original"]):
        return context["cache"][("baseline",)]
    y = context["baseline_input"] if baseline else context["input100"] + (scale-1.)*context["edge_original"]
    prediction = np.full(len(y), np.nan)
    geometry = er_matrix_geometry(context["X"][mask]) if baseline else context["masks"][fit_radius_mm][1]
    diagnostic = dict(context["identity"], axis=context["axis"], n_eval=len(y), n_train=int(mask.sum()), n_excluded=int((~mask).sum()), n_parameters=len(context["names"]),
        rank=geometry["rank"], scaled_condition=geometry["scaled_condition"], rank_deficient=geometry["rank"] < len(context["names"]),
        few_points_flag=int(mask.sum()) < context["options"]["min_points_guide"], status="no_training_points", method=None, gopt=np.nan, lambda_max=np.nan, span_iter=0, span_violation_count=0)
    coefficient = dict(context["identity"], axis=context["axis"])
    if mask.any():
        options = context["options"]
        try:
            fit = ep_solve_vba_style_ridge_one_axis(X_fit=context["X"][mask], y=y[mask], X_eval=context["Xv"], coeff_names=context["names"], span_limits=ep_get_active_span_limits(options["span_mode"]),
                ridge_regression=options["ridge_regression"], gopt_spec=options["gopt_spec"], gopt_tolerance=options["gopt_tolerance"], minimum_ridge=EP_MINIMUM_RIDGE,
                add_ridge=EP_ADD_RIDGE, max_gopt_return=EP_MAX_GOPT_RETURN, gopt_type=options["gopt_type"], inner_or_partial=ep_determine_inner_or_partial(int(mask.sum()), context["n_virtual"]),
                use_span_box_constraint=options["use_span_box_constraint"], span_tolerance=EP_SPAN_TOLERANCE, cd_tol=EP_CD_TOL, max_span_iter=EP_MAX_SPAN_ITER,
                base_lambda_scale=ep_choose_base_lambda_scale(int(mask.sum())))
            prediction = context["X"] @ fit.beta  # 146~147mm처럼 회귀에서 제외한 point도 같은 모델로 평가
            warnings = []
            if not np.isfinite(prediction).all():
                prediction[:] = np.nan
                warnings.append("nonfinite_prediction")
            if fit.violations:
                warnings.append("span_violations")
            if options["ridge_regression"] and (not np.isfinite(fit.gopt) or fit.gopt > options["gopt_spec"]+options["gopt_tolerance"]):
                warnings.append("gopt_not_met")
            diagnostic.update(status="|".join(warnings) if warnings else "ok", method=fit.method, gopt=fit.gopt, lambda_max=float(np.max(fit.lambdas)),
                              span_iter=fit.span_iter, span_violation_count=len(fit.violations), virtual_source=context["virtual_source"], n_virtual=context["n_virtual"])
            for name, value, ridge in zip(context["names"], fit.beta, fit.lambdas):
                coefficient[name], coefficient[f"lambda_{name}"] = float(value), float(ridge)
        except (ValueError, np.linalg.LinAlgError, FloatingPointError) as exc:
            diagnostic.update(status="solver_error", error_message=str(exc))
    solution = dict(prediction=prediction, diagnostic=diagnostic, coefficient=coefficient)
    context["cache"][cache_key] = solution
    return solution


def er_check_baseline(points, prediction, axis, product_number):
    rows = []
    for column, calculated in [(f"CPE_fit_{axis}", prediction), (f"real_residual_{axis}", points[f"raw_minus_total_fit_{axis}"]-prediction)]:
        if column not in points:
            rows.append(dict(column=column, check_status="not_in_input", passed=True, n_checked=0, max_abs_error_nm=np.nan))
            continue
        old, new = pd.to_numeric(points[column], errors="coerce").to_numpy(float), calculated.to_numpy(float)
        valid = np.isfinite(old) & np.isfinite(new)
        passed = np.allclose(old, new, rtol=1e-7, atol=ER_BASELINE_ATOL_UM, equal_nan=True)
        rows.append(dict(column=column, check_status="checked", passed=bool(passed), n_checked=int(valid.sum()), max_abs_error_nm=float(np.max(np.abs(old[valid]-new[valid]))*1000) if valid.any() else np.nan))
    table = pd.DataFrame(rows)
    if not table.passed.all():
        path = EP_OUTPUT_DIR / f"Baseline_Parity_Failed_Product_{product_number:02d}.xlsx"
        table.to_excel(path, index=False)
        raise ValueError(f"저장된 27번 CPE와 기준 재계산이 다릅니다. 같은 실행의 27번/28번 설정인지, 비선택 방향 결측 또는 147mm 밖 회귀점이 있었는지 확인하세요: {path}")
    return table


def er_metric_table(frame, keys, skeleton=None):
    expected = frame.loc[frame._fit_valid]
    result = ep_group_stats(expected, keys, "_metric_residual", skeleton=frame if skeleton is None else skeleton)
    result = result.rename(columns={"n_points": "n_expected", "n_valid": "n_predicted"})
    result["coverage_fraction"] = result.n_predicted/result.n_expected.replace(0, np.nan)
    incomplete = result.n_predicted.lt(result.n_expected)
    columns = [c for c in EP_STAT_COLUMNS if c not in ["n_points", "n_valid"]]
    result.loc[incomplete, columns] = np.nan  # 예측 누락을 제거해 통계가 좋아 보이는 것을 방지
    result["metric_status"] = np.where(result.n_expected.eq(0), "no_valid_input", np.where(incomplete, "incomplete_prediction", np.where(result.n_predicted.le(EP_STD_DDOF), "too_few_points", "ok")))
    return result


def er_scenario_metrics(frame, radius, scale):
    wafer_tables, shot_tables = [], []
    regions = {"R_LE_144MM": frame.radius.le(EP_INNER_RADIUS_UM), "ALL_R_LE_147MM": pd.Series(True, index=frame.index)}
    for scope, selected in [("ALL_SHOTS", pd.Series(True, index=frame.index)), ("MIXED_SHOTS", frame._mixed_shot)]:
        for region, mask in regions.items():
            table = er_metric_table(frame.loc[selected & mask], EP_WAFER_KEYS, skeleton=frame)
            table["scope"], table["region"] = scope, region
            wafer_tables.append(table)
    edge_shots = frame.loc[frame._has_edge_shot]
    for region, mask in [("R_LE_144MM", edge_shots.radius.le(EP_INNER_RADIUS_UM)), ("ALL_R_LE_147MM", pd.Series(True, index=edge_shots.index))]:
        table = er_metric_table(edge_shots.loc[mask], EP_SHOT_KEYS, skeleton=edge_shots)
        table["region"] = region
        shot_tables.append(table)
    wafer, shot = pd.concat(wafer_tables, ignore_index=True), pd.concat(shot_tables, ignore_index=True)
    for table in [wafer, shot]:
        table["fit_radius_mm"], table["scale"], table["scale_pct"] = radius, scale, int(round(scale*100))
    return wafer, shot


## 05. 실제 point 비율·동일 mixed shot 입력 차이


In [ ]:
def er_shot_metadata(points):
    cols = [c for c in ["DieX", "DieY", "photo_transn_seq"] if c in points]
    return points.groupby(EP_SHOT_KEYS, observed=True, dropna=False)[cols].first().reset_index() if cols else points[EP_SHOT_KEYS].drop_duplicates()


def er_point_counts(points, keys):
    work = points[keys].copy()
    work["n_total"] = 1
    work["n_inner"] = points.radius.le(EP_INNER_RADIUS_UM).astype(int)
    work["n_edge"] = points.radius.gt(EP_INNER_RADIUS_UM).astype(int)
    work["n_edge_to_lower_radius"] = (points.radius.gt(EP_INNER_RADIUS_UM) & points.radius.le(min(ER_FIT_RADII_MM)*1000)).astype(int)
    work["n_above_lower_radius"] = points.radius.gt(min(ER_FIT_RADII_MM)*1000).astype(int)
    work["n_valid"] = points._fit_valid.astype(int)
    work["n_valid_inner"] = (points._fit_valid & points.region.eq("INNER")).astype(int)
    work["n_valid_edge"] = (points._fit_valid & points.region.eq("EDGE")).astype(int)
    work["n_inner_in_mixed_shots"] = (points._fit_valid & points.region.eq("INNER") & points._mixed_shot).astype(int)
    if "measurement_source" in points:
        work["n_edge_source_points"] = points.measurement_source.eq("oco_edge").astype(int)
    columns = [col for col in work if col not in keys]
    result = work.groupby(keys, observed=True, dropna=False)[columns].sum().reset_index()
    result["edge_point_fraction"] = result.n_edge/result.n_total.replace(0, np.nan)
    result["valid_edge_fraction"] = result.n_valid_edge/result.n_valid.replace(0, np.nan)
    result["inner_mixed_fraction"] = result.n_inner_in_mixed_shots/result.n_valid_inner.replace(0, np.nan)
    result["lower_fit_radius_mm"] = min(ER_FIT_RADII_MM)
    return result


def er_mixed_input_gap(points, keys, axis, skeleton):
    selected = points.loc[points._mixed_shot & points._fit_valid]
    tables = []
    for region in ["INNER", "EDGE"]:
        table = ep_group_stats(selected.loc[selected.region.eq(region)], keys, f"input100_{axis}", skeleton=skeleton)
        table["axis"], table["region"] = axis, region
        tables.append(table)
    result = ep_region_comparison(pd.concat(tables, ignore_index=True), keys)
    result["comparable"] = result.n_valid_inner.gt(EP_STD_DDOF) & result.n_valid_edge.gt(EP_STD_DDOF)
    return result


## 06. 제품별 실행: 146/147mm × 10~100%


In [ ]:
er_wafer_metrics, er_shot_metrics, er_wafer_counts, er_edge_shot_counts, er_shot_counts = [], [], [], [], []
er_wafer_gaps, er_shot_gaps, er_parity, er_audits, er_diagnostic_summaries, er_sources = [], [], [], [], [], []
for product_number, source in enumerate(EP_SOURCES, start=1):
    product, axis = str(source["product"]), str(source["axis"]).lower()
    raw, saved_settings, point_path, settings_path = ep_read_product(source)
    options = ep_restore_settings(saved_settings)
    er_cached_geometry_gopt.cache_clear()
    points, audit = er_prepare_product(raw, product, axis)
    del raw
    metadata = er_shot_metadata(points)
    counts = er_point_counts(points, EP_WAFER_KEYS)
    edge_counts = er_point_counts(points.loc[points._has_edge_shot], EP_WAFER_KEYS)
    shot_counts = er_point_counts(points, EP_SHOT_KEYS).merge(metadata, on=EP_SHOT_KEYS, how="left", validate="1:1")
    wafer_gap = er_mixed_input_gap(points, EP_WAFER_KEYS, axis, skeleton=points)
    shot_gap = er_mixed_input_gap(points, EP_SHOT_KEYS, axis, skeleton=points.loc[points._mixed_shot]).merge(metadata, on=EP_SHOT_KEYS, how="left", validate="1:1")
    for table in [counts, edge_counts, shot_counts]:
        table["axis"] = axis
    er_wafer_counts.append(counts)
    er_edge_shot_counts.append(edge_counts)
    er_shot_counts.append(shot_counts)
    er_wafer_gaps.append(wafer_gap)
    er_shot_gaps.append(shot_gap)
    er_audits.append(audit)
    er_sources.append(dict(product=product, axis=axis, reference_scale=source["reference_scale"], points=str(point_path), settings=str(settings_path), original_settings=saved_settings))
    contexts = er_build_contexts(points, axis, options)
    baseline_prediction = pd.Series(np.nan, index=points.index, dtype=float)
    baseline_diagnostics, baseline_coefficients = [], []
    for context in contexts:
        solution = er_solve_context(context, max(ER_FIT_RADII_MM), scale=None)
        baseline_prediction.loc[context["indices"]] = solution["prediction"]
        baseline_diagnostics.append(solution["diagnostic"])
        baseline_coefficients.append(solution["coefficient"])
    parity = er_check_baseline(points, baseline_prediction, axis, product_number)
    parity["product"], parity["axis"] = product, axis
    er_parity.append(parity)
    pd.DataFrame(baseline_diagnostics).to_parquet(EP_OUTPUT_DIR / "diagnostics" / f"Product_{product_number:02d}_baseline.parquet", index=False)
    if ER_SAVE_COEFFICIENTS:
        pd.DataFrame(baseline_coefficients).to_parquet(EP_OUTPUT_DIR / "coefficients" / f"Product_{product_number:02d}_baseline.parquet", index=False)
    point_cols = EP_SHOT_KEYS + [c for c in ["DieX", "DieY", "photo_transn_seq", "TEST", "GROUP", "coordinate_X", "coordinate_Y", "radius", "measurement_source", "source_mstepseq", "measurement_scale", f"{axis.upper()}_reg_original", f"raw_minus_total_fit_{axis}", f"input100_{axis}"] if c in points]
    baseline_points = points[point_cols].copy()
    baseline_points["analysis_axis"] = axis
    baseline_points[f"CPE_fit_{axis}"] = baseline_prediction
    baseline_points[f"metric_residual_{axis}"] = points[f"input100_{axis}"]-baseline_prediction
    baseline_points.to_parquet(EP_OUTPUT_DIR / "points" / f"Product_{product_number:02d}_baseline_all_points.parquet", index=False)
    frame = points[EP_SHOT_KEYS + ["radius", "_fit_valid", "_mixed_shot", "_has_edge_shot"]].copy()
    affected = [context for context in contexts if context["affected"]]
    print(f"{product} [{axis.upper()}]: wafer {len(counts):,}, 전체 shot {len(contexts):,}, 영향 shot {len(affected):,}; 기준 CPE 검증 완료")
    for radius in ER_FIT_RADII_MM:
        for scale in ER_SCALES:
            prediction = baseline_prediction.copy()
            diagnostics, coefficients = [], []
            for context in affected:
                solution = er_solve_context(context, radius, scale=scale)
                prediction.loc[context["indices"]] = solution["prediction"]
                case = dict(fit_radius_mm=radius, scale=scale, scale_pct=int(round(scale*100)))
                diagnostics.append(dict(solution["diagnostic"], **case))
                coefficients.append(dict(solution["coefficient"], **case))
            frame["_metric_residual"] = points[f"input100_{axis}"]-prediction  # 평가는 항상 배율 적용 전 값
            wafer, shot = er_scenario_metrics(frame, radius, scale)
            wafer["axis"], shot["axis"] = axis, axis
            shot = shot.merge(metadata, on=EP_SHOT_KEYS, how="left", validate="m:1")
            er_wafer_metrics.append(wafer)
            er_shot_metrics.append(shot)
            stem = f"Product_{product_number:02d}_R{radius:g}_S{int(round(scale*100)):03d}"
            wafer.to_parquet(EP_OUTPUT_DIR / "cases" / f"{stem}_Wafer_M3S.parquet", index=False)
            shot.to_parquet(EP_OUTPUT_DIR / "cases" / f"{stem}_Shot_M3S.parquet", index=False)
            diag = pd.DataFrame(diagnostics) if diagnostics else pd.DataFrame(columns=EP_SHOT_KEYS+["axis", "status", "rank_deficient", "few_points_flag", "fit_radius_mm", "scale", "scale_pct"])
            diag.to_parquet(EP_OUTPUT_DIR / "diagnostics" / f"{stem}.parquet", index=False)
            if not diag.empty:
                summary = diag.groupby(["product", "axis", "status", "rank_deficient", "few_points_flag"], observed=True, dropna=False).size().rename("n_shots").reset_index()
                summary["fit_radius_mm"], summary["scale_pct"] = radius, int(round(scale*100))
                er_diagnostic_summaries.append(summary)
            if ER_SAVE_COEFFICIENTS:
                pd.DataFrame(coefficients).to_parquet(EP_OUTPUT_DIR / "coefficients" / f"{stem}.parquet", index=False)
            if ER_SAVE_SCENARIO_POINTS:
                selected = points._has_edge_shot
                trace = points.loc[selected, point_cols].copy()
                trace["analysis_axis"], trace["fit_radius_mm"], trace["edge_scale"] = axis, radius, scale
                trace["used_for_fit"] = points.loc[selected, "_fit_valid"] & points.loc[selected, "radius"].le(radius*1000)
                trace[f"CPE_fit_{axis}"] = prediction.loc[selected]
                trace[f"metric_residual_{axis}"] = frame.loc[selected, "_metric_residual"]
                original = pd.Series(0., index=trace.index)
                outer = trace.radius.gt(EP_INNER_RADIUS_UM) & points.loc[selected, "_fit_valid"]
                if outer.any():
                    original.loc[outer] = trace.loc[outer, f"{axis.upper()}_reg_original"]
                trace[f"training_input_{axis}"] = trace[f"input100_{axis}"] + (scale-1.)*original
                trace[f"training_residual_{axis}"] = (trace[f"training_input_{axis}"]-trace[f"CPE_fit_{axis}"]).where(trace.used_for_fit)
                trace.to_parquet(EP_OUTPUT_DIR / "points" / f"{stem}.parquet", index=False)
            print(f"  fit R<={radius:g}mm / edge {scale:.0%}: inner M3S 계산, CPE 진단 {len(diag):,} shots")
    del points, contexts, frame, baseline_points

er_wafer_metrics = pd.concat(er_wafer_metrics, ignore_index=True)
er_shot_metrics = pd.concat(er_shot_metrics, ignore_index=True)
er_wafer_counts = pd.concat(er_wafer_counts, ignore_index=True)
er_edge_shot_counts = pd.concat(er_edge_shot_counts, ignore_index=True)
er_shot_counts = pd.concat(er_shot_counts, ignore_index=True)
er_wafer_gaps = pd.concat(er_wafer_gaps, ignore_index=True)
er_shot_gaps = pd.concat(er_shot_gaps, ignore_index=True)
er_parity = pd.concat(er_parity, ignore_index=True)
er_input_audit = pd.DataFrame(er_audits)
er_diagnostic_summary = pd.concat(er_diagnostic_summaries, ignore_index=True) if er_diagnostic_summaries else pd.DataFrame(columns=["product", "axis", "status", "rank_deficient", "few_points_flag", "n_shots", "fit_radius_mm", "scale_pct"])
display(er_parity)
display(er_input_audit)


## 07. 같은 wafer 집합의 비교표·최적 배율


In [ ]:
er_cohort_keys = EP_WAFER_KEYS + ["axis", "scope", "region"]
er_case_count = len(ER_FIT_RADII_MM)*len(ER_SCALES)
er_completeness = er_wafer_metrics.groupby(er_cohort_keys, observed=True, dropna=False).M3S_nm.count().rename("n_finite_cases").reset_index()
er_completeness["complete_all_cases"] = er_completeness.n_finite_cases.eq(er_case_count)
er_wafer_metrics = er_wafer_metrics.merge(er_completeness, on=er_cohort_keys, how="left", validate="m:1")
curve_keys = ["product", "axis", "scope", "region", "fit_radius_mm", "scale", "scale_pct"]
er_product_curves = er_wafer_metrics[curve_keys].drop_duplicates()
er_curve_population = er_wafer_metrics.groupby(curve_keys, observed=True, dropna=False).size().rename("n_wafers_total").reset_index()
complete = er_wafer_metrics.loc[er_wafer_metrics.complete_all_cases]
if not complete.empty:
    grouped = complete.groupby(curve_keys, observed=True, dropna=False).M3S_nm
    summary = grouped.agg(n_wafers_used="count", mean_M3S_nm="mean", median_M3S_nm="median", min_M3S_nm="min", max_M3S_nm="max").reset_index()
    er_product_curves = er_product_curves.merge(summary, on=curve_keys, how="left", validate="1:1")
else:
    er_product_curves = er_product_curves.assign(n_wafers_used=0, mean_M3S_nm=np.nan, median_M3S_nm=np.nan, min_M3S_nm=np.nan, max_M3S_nm=np.nan)
er_product_curves = er_product_curves.merge(er_curve_population, on=curve_keys, how="left", validate="1:1")
er_product_curves["n_wafers_used"] = er_product_curves.n_wafers_used.fillna(0).astype(int)

delta_keys = er_cohort_keys + ["scale", "scale_pct"]
columns = delta_keys + ["M3S_nm", "abs_mean_nm", "three_sigma_nm", "n_expected", "n_predicted", "metric_status"]
small = er_wafer_metrics.loc[er_wafer_metrics.fit_radius_mm.eq(min(ER_FIT_RADII_MM)), columns]
large = er_wafer_metrics.loc[er_wafer_metrics.fit_radius_mm.eq(max(ER_FIT_RADII_MM)), columns]
er_radius_delta = small.merge(large, on=delta_keys, suffixes=("_smaller_radius", "_larger_radius"), validate="1:1")
for col in ["M3S_nm", "abs_mean_nm", "three_sigma_nm"]:
    er_radius_delta[f"delta_{col}"] = er_radius_delta[f"{col}_smaller_radius"]-er_radius_delta[f"{col}_larger_radius"]
er_radius_delta["smaller_fit_radius_mm"], er_radius_delta["larger_fit_radius_mm"] = min(ER_FIT_RADII_MM), max(ER_FIT_RADII_MM)

best_keys = er_cohort_keys + ["fit_radius_mm"]
er_best_scales = er_wafer_metrics[best_keys].drop_duplicates()
counts = er_wafer_metrics.groupby(best_keys, observed=True, dropna=False).M3S_nm.count().rename("n_finite_scales").reset_index()
er_best_scales = er_best_scales.merge(counts, on=best_keys, validate="1:1")
valid = er_wafer_metrics.merge(counts, on=best_keys, validate="m:1")
valid = valid.loc[valid.n_finite_scales.eq(len(ER_SCALES))]
if not valid.empty:
    indices = valid.groupby(best_keys, observed=True, dropna=False).M3S_nm.idxmin()
    best = valid.loc[indices, best_keys+["scale_pct", "M3S_nm"]].rename(columns={"scale_pct":"best_scale_pct", "M3S_nm":"best_M3S_nm"})
    er_best_scales = er_best_scales.merge(best, on=best_keys, how="left", validate="1:1")
else:
    er_best_scales = er_best_scales.assign(best_scale_pct=np.nan, best_M3S_nm=np.nan)
hundred = er_wafer_metrics.loc[er_wafer_metrics.scale.eq(1.), best_keys+["M3S_nm"]].rename(columns={"M3S_nm":"M3S_100pct_nm"})
er_best_scales = er_best_scales.merge(hundred, on=best_keys, how="left", validate="1:1")
er_best_scales["loss_100_minus_wafer_best_nm"] = er_best_scales.M3S_100pct_nm-er_best_scales.best_M3S_nm

er_reference_scales = {str(s["product"]):float(s["reference_scale"]) for s in EP_SOURCES}
reference_rows = er_wafer_metrics.loc[np.isclose(er_wafer_metrics.scale, er_wafer_metrics["product"].map(er_reference_scales))]
reference = reference_rows[best_keys+["M3S_nm", "abs_mean_nm", "three_sigma_nm"]].rename(columns={"M3S_nm":"M3S_reference_nm", "abs_mean_nm":"abs_mean_reference_nm", "three_sigma_nm":"three_sigma_reference_nm"})
hundred = er_wafer_metrics.loc[er_wafer_metrics.scale.eq(1.), best_keys+["M3S_nm", "abs_mean_nm", "three_sigma_nm"]].rename(columns={"M3S_nm":"M3S_100pct_nm", "abs_mean_nm":"abs_mean_100pct_nm", "three_sigma_nm":"three_sigma_100pct_nm"})
er_reference_loss = reference.merge(hundred, on=best_keys, validate="1:1")
er_reference_loss["reference_scale_pct"] = er_reference_loss["product"].map(er_reference_scales)*100
er_reference_loss["loss_100_minus_reference_nm"] = er_reference_loss.M3S_100pct_nm-er_reference_loss.M3S_reference_nm
er_reference_loss["delta_abs_mean_nm"] = er_reference_loss.abs_mean_100pct_nm-er_reference_loss.abs_mean_reference_nm
er_reference_loss["delta_three_sigma_nm"] = er_reference_loss.three_sigma_100pct_nm-er_reference_loss.three_sigma_reference_nm
er_reference_loss = er_reference_loss.merge(er_wafer_counts, on=EP_WAFER_KEYS+["axis"], how="left", validate="m:1")
er_reference_loss = er_reference_loss.merge(er_wafer_gaps[EP_WAFER_KEYS+["axis", "delta_M3S_edge_minus_inner_nm"]], on=EP_WAFER_KEYS+["axis"], how="left", validate="m:1")

er_wafer_index = er_wafer_counts[EP_WAFER_KEYS+["axis"]].drop_duplicates().sort_values(EP_WAFER_KEYS).reset_index(drop=True)
er_wafer_index["wafer_index"] = er_wafer_index.groupby("product", observed=True).cumcount()
er_wafer_metrics = er_wafer_metrics.merge(er_wafer_index, on=EP_WAFER_KEYS+["axis"], how="left", validate="m:1")
inner_shot = er_shot_metrics.loc[er_shot_metrics.region.eq("R_LE_144MM")].copy()
inner_shot["case"] = [f"R{r:g}_S{int(s):03d}" for r,s in zip(inner_shot.fit_radius_mm,inner_shot.scale_pct)]
er_shot_inner_wide = inner_shot.pivot(index=EP_SHOT_KEYS+["axis"], columns="case", values="M3S_nm").reset_index()
er_shot_inner_wide.columns.name = None
er_shot_inner_wide = er_shot_inner_wide.merge(er_shot_counts[EP_SHOT_KEYS+["axis"]+[c for c in ["DieX", "DieY", "photo_transn_seq"] if c in er_shot_counts]], on=EP_SHOT_KEYS+["axis"], how="left", validate="1:1")
er_sampling_summary = er_wafer_counts.groupby(["product", "axis"], observed=True).agg(n_wafers=("Wafer","size"), median_edge_point_fraction=("edge_point_fraction","median"), median_inner_mixed_fraction=("inner_mixed_fraction","median"), total_measured_points=("n_total","sum"), total_edge_points=("n_edge","sum")).reset_index()
display(er_sampling_summary)
display(er_product_curves.loc[er_product_curves.scope.eq("ALL_SHOTS") & er_product_curves.region.eq("R_LE_144MM")])


## 08. 제품·wafer 비교 그래프


In [ ]:
er_products = [str(s["product"]) for s in EP_SOURCES]
er_labels = {p:f"{p} [{EP_PRODUCT_AXES[p][0].upper()}]" for p in er_products}
er_colors = dict(zip(er_products, plt.get_cmap("tab10")(np.linspace(0, .9, len(er_products)))))
er_scale_colors = dict(zip(ER_SCALES, plt.get_cmap("tab10")(np.linspace(0, .9, len(ER_SCALES)))))
er_plot_paths = []


def er_save_plot(fig, name):
    for extension in ["png", "pdf"]:
        path = EP_OUTPUT_DIR / "plots" / f"{name}.{extension}"
        fig.savefig(path, dpi=170, bbox_inches="tight")
        er_plot_paths.append(path)
    display(fig)
    plt.close(fig)


def er_boxes(panel, data, column, title, ylabel, unit="wafers"):
    rng = np.random.default_rng(20261029)
    labels = []
    for index, product in enumerate(er_products):
        values = pd.to_numeric(data.loc[data["product"].eq(product),column],errors="coerce").dropna().to_numpy(float)
        labels.append(f"{er_labels[product]}\nN={len(values)} {unit}")
        if len(values):
            panel.boxplot([values],positions=[index],widths=.5,patch_artist=True,showfliers=False,boxprops=dict(facecolor=er_colors[product],alpha=.3),medianprops=dict(color="black"))
            panel.scatter(index+rng.uniform(-.12,.12,len(values)),values,s=10,alpha=.4,color=er_colors[product])
        else:
            panel.text(index,.5,"No comparable data",ha="center",rotation=90,transform=panel.get_xaxis_transform())
    panel.set_xticks(np.arange(len(er_products)),labels)
    panel.set_xlim(-.6,len(er_products)-.4)
    panel.set(title=title,ylabel=ylabel)
    panel.grid(axis="y",alpha=.25)


fig, panels = plt.subplots(1,2,figsize=(14,5))
er_boxes(panels[0],er_wafer_counts,"edge_point_fraction","Measured edge points / all wafer points","Point fraction")
er_boxes(panels[1],er_edge_shot_counts,"edge_point_fraction","Edge points / points in shots containing edge","Point fraction")
fig.tight_layout()
er_save_plot(fig,"01_Actual_edge_point_fraction")

fig, panels = plt.subplots(1,2,figsize=(14,5))
er_boxes(panels[0],er_wafer_gaps,"delta_M3S_edge_minus_inner_nm","Mixed shots pooled per wafer: edge M3S - inner M3S","M3S difference [nm]")
er_boxes(panels[1],er_shot_gaps,"delta_M3S_edge_minus_inner_nm","Same mixed shot: edge M3S - inner M3S","M3S difference [nm]",unit="shots")
for panel in panels:
    panel.axhline(0,color="gray",linewidth=.8)
fig.suptitle("Unscaled input after fixed K-MRC, before CPE")
fig.tight_layout()
er_save_plot(fig,"02_Mixed_shot_input_gap")

for scope,region,name in [("ALL_SHOTS","R_LE_144MM","03_Product_M3S_Inner"),("MIXED_SHOTS","R_LE_144MM","04_Product_M3S_Mixed_Inner"),("ALL_SHOTS","ALL_R_LE_147MM","05_Product_M3S_All147")]:
    fig, panels = plt.subplots(1,len(er_products),figsize=(5*len(er_products),4.5),squeeze=False,sharey=True)
    for panel,product in zip(panels[0],er_products):
        part = er_product_curves.loc[er_product_curves["product"].eq(product)&er_product_curves.scope.eq(scope)&er_product_curves.region.eq(region)]
        for radius in ER_FIT_RADII_MM:
            line = part.loc[part.fit_radius_mm.eq(radius)].sort_values("scale_pct")
            panel.plot(line.scale_pct,line.mean_M3S_nm,".-",label=f"Fit R <= {radius:g} mm; N={int(line.n_wafers_used.max()) if len(line) else 0}")
        if part.mean_M3S_nm.notna().sum()==0:
            panel.text(.5,.5,"No wafers complete across all cases",ha="center",wrap=True,transform=panel.transAxes)
        panel.set(title=er_labels[product],xlabel="Edge value scale for CPE fit [%]",ylabel="Mean wafer M3S [nm]")
        panel.grid(alpha=.25)
        panel.legend(fontsize=8)
    fig.suptitle(f"{scope} | evaluation {region} | same wafer cohort, unscaled evaluation")
    fig.tight_layout()
    er_save_plot(fig,name)

fig, panels = plt.subplots(1,len(er_products),figsize=(5*len(er_products),4.5),squeeze=False,sharey=True)
complete_delta = er_radius_delta.merge(er_completeness,on=er_cohort_keys,how="left",validate="m:1")
for panel,product in zip(panels[0],er_products):
    part = complete_delta.loc[complete_delta["product"].eq(product)&complete_delta.scope.eq("ALL_SHOTS")&complete_delta.region.eq("R_LE_144MM")&complete_delta.complete_all_cases]
    grouped = part.groupby("scale_pct",observed=True).delta_M3S_nm
    mean = grouped.mean()
    panel.plot(mean.index,mean.to_numpy(),".-",color=er_colors[product])
    panel.axhline(0,color="gray",linewidth=.8)
    panel.set(title=er_labels[product],xlabel="Edge value scale [%]",ylabel="M3S: smaller fit radius - larger [nm]")
    panel.grid(alpha=.25)
fig.suptitle("Same-wafer radius difference | R <= 144 mm evaluation | negative = smaller radius better")
fig.tight_layout()
er_save_plot(fig,"06_Paired_radius_difference")

fig, panels = plt.subplots(1,2,figsize=(14,5))
relation = er_reference_loss.loc[er_reference_loss.scope.eq("ALL_SHOTS")&er_reference_loss.region.eq("R_LE_144MM")&er_reference_loss.fit_radius_mm.eq(max(ER_FIT_RADII_MM))]
for product in er_products:
    part=relation.loc[relation["product"].eq(product)]
    label=f"{er_labels[product]} (reference {er_reference_scales[product]:.0%})"
    for panel,column in zip(panels,["valid_edge_fraction","delta_M3S_edge_minus_inner_nm"]):
        paired=part.dropna(subset=[column,"loss_100_minus_reference_nm"])
        panel.scatter(paired[column],paired.loss_100_minus_reference_nm,color=er_colors[product],s=20,alpha=.6,label=label)
for panel in panels:
    panel.axhline(0,color="gray",linewidth=.8)
    panel.set_ylabel("Inner M3S: 100% - previous product reference [nm]")
    panel.grid(alpha=.25)
    panel.legend(title="Product / axis",fontsize=8)
panels[0].set_xlabel("Valid edge points / valid wafer points")
panels[1].set_xlabel("Mixed-shot pooled input M3S: edge - inner [nm]")
fig.suptitle("Same-wafer association | larger fit radius | reference scales are fixed per product")
fig.tight_layout()
er_save_plot(fig,"07_Input_gap_fraction_vs_100pct_loss")

from matplotlib.ticker import MaxNLocator
for index,product in enumerate(er_products,start=1):
    fig, panels = plt.subplots(2,2,figsize=(17,9),sharex=True,sharey="row")
    for row,region in enumerate(["R_LE_144MM","ALL_R_LE_147MM"]):
        for col,radius in enumerate(ER_FIT_RADII_MM):
            panel=panels[row,col]
            for scale in ER_SCALES:
                part=er_wafer_metrics.loc[er_wafer_metrics["product"].eq(product)&er_wafer_metrics.scope.eq("ALL_SHOTS")&er_wafer_metrics.region.eq(region)&er_wafer_metrics.fit_radius_mm.eq(radius)&er_wafer_metrics.scale.eq(scale)].sort_values("wafer_index")
                panel.plot(part.wafer_index,part.M3S_nm,".-",linewidth=.7,markersize=2,color=er_scale_colors[scale],label=f"{scale:.0%}")
            panel.set(title=f"Fit <= {radius:g} mm | evaluate {region}",ylabel="|mean| + 3 sigma [nm]")
            panel.grid(alpha=.25)
            panel.xaxis.set_major_locator(MaxNLocator(integer=True))
            if row==0:
                panel.legend(title="Edge fit scale",ncol=5,fontsize=8)
            if row==1:
                panel.set_xlabel("Wafer index (see Wafer_index sheet)")
    fig.suptitle(f"{er_labels[product]} | unscaled evaluation on original measured points")
    fig.tight_layout()
    er_save_plot(fig,f"08_Wafer_M3S_Product_{index:02d}")


## 09. Excel·전체 Parquet·설정 저장


In [ ]:
er_notes = pd.DataFrame([
    ["입력", "제품별 27_OCO_FINAL_POINTS 전체 Parquet/Excel + 같은 실행의 extension_settings.json. sample 파일 미사용. DB 조회 없음."],
    ["방향", "; ".join(f"{p}: {a[0].upper()}" for p,a in EP_PRODUCT_AXES.items())],
    ["CPE", "기존 X21/Y17 및 제품별 ridge/G-opt/span/virtual 설정. K-MRC·기적용 MRC/PSM decorrection은 고정. 회귀 point만 반경으로 제한."],
    ["배율", "144mm 초과의 원래 측정 X/Y 값에 10~100% 적용. CPE 입력=복원 input100+(배율-1)*원래 측정값. raw_minus_total_fit 자체에 배율을 곱하지 않음."],
    ["평가", "metric_residual=input100-CPE_fit. 모든 배율에서 평가 입력은 동일한 원래 값. 146~147mm도 회귀에서만 제외하고 평가에는 유지."],
    ["두 평가 영역", "R_LE_144MM: 공식 사용 메트릭. ALL_R_LE_147MM: 원래 반경 147mm 이하 모든 유효 측정점. 반경 미상/147mm 초과 제외 수는 Input_audit."],
    ["두 shot 범위", "ALL_SHOTS: 전체 wafer. MIXED_SHOTS: 144mm 안쪽과 바깥쪽 유효 측정점이 함께 있는 shot만. 같은 shot 집합을 모든 조건에서 유지."],
    ["Edge shot 선정", "실제 point 반경 >144mm가 있는 shot. 077252가 144mm 밖을 측정하는 현재 구성과 같은 선정. 단계명이 달라도 반경 기준으로 동작. 실제 measurement_source 개수도 있으면 기록."],
    ["실제 point 비율", "edge_point_fraction=측정 edge point/전체 측정 point. valid_edge_fraction은 선택 방향과 좌표가 유효한 point 기준. 이전 eligible/all shot 비율과 다름."],
    ["Mixed 입력 차이", "CPE 제거 전 100% 복원 입력의 같은 shot 안쪽/edge M3S 차이. Wafer 차이는 mixed shot들의 point를 모아 다시 계산. Shot M3S의 평균이 아님."],
    ["유효 mask", "선택한 방향의 input100 및 X/Y 좌표 유효. 비선택 방향의 결측은 무시. 제품별 현재 mask로 기준 재계산이 기존 27번과 다르면 중단하여 진단 파일 저장."],
    ["기준 검증", "원래 measurement_scale 그대로 CPE를 한 번 재계산. 입력에 CPE_fit/real_residual이 있으면 일치 검사. 컬럼이 없으면 not_in_input으로 기록하며 재현 검증을 주장하지 않음."],
    ["저표본 shot", "Para 축소 없이 기존 ridge 방식을 사용하며 n_train/rank/few_points/G-opt/span 상태 기록. 안쪽 참조용 point 부족 screening으로 CPE shot을 임의 제외하지 않음."],
    ["학습점 0개", "모델 재사용 없이 예측 NA. 평가 region에 예측 누락이 있으면 그 그룹 M3S도 NA(incomplete_prediction). 예측 가능한 일부 point로 축소해 개선처럼 표시하지 않음."],
    ["제품 평균 곡선", "각 scope/평가 영역에서 모든 반경×배율 조건의 M3S가 유효한 같은 wafer 집합만 평균. n_wafers_used/total 함께 기록. Wafer별 원자료는 그대로 보존."],
    ["M3S", f"abs(mean)+3*std, ddof={EP_STD_DDOF}. 최종 residual point를 모아 계산. mean(abs)나 shot 평균의 평균이 아님. 통계/그래프 nm, point 파일 µm."],
    ["반경 차이", "Radius_delta: 작은 회귀 반경 M3S - 큰 회귀 반경 M3S, 같은 wafer·배율·평가 영역. 음수면 작은 회귀 반경이 유리."],
    ["기존 최적값", "Previous_opt_loss: 제품별 reference_scale(VM50/VF20/VC40) 고정 대비 100% 차이. Wafer_best_scale은 별도로 각 wafer의 시험 후보 중 최소값(동률은 낮은 배율)."],
    ["해석", "146mm 제한은 point 수와 최외곽 공간 영역을 함께 바꿈. 개선을 순수 point 개수 효과로 단정하지 않음."],
    ["가상 grid", "제품별 저장된 virtual grid와 wafer clip 유지. Fallback grid도 원래 전체 shot 측정 범위로 고정하여 회귀 반경과 평가 범위 혼동 방지."],
    ["최적화", "Edge 없는 shot은 기준 CPE 재사용. 두 반경의 학습점이 같은 shot은 배율별 해 재사용. 좌표/lambda 기반 G-opt만 bounded cache; y 의존 계수/span은 재계산."],
    ["Excel 용량", "Shot_M3S 전체 long table과 전체 Shot_point_counts는 tables Parquet에 저장. Excel에는 edge shot count와 shot별 inner M3S 20조건 wide table을 제공. sample로 대체하지 않음."],
    ["Point 파일", "Product_NN_baseline_all_points는 전체 wafer 기준. 반경/배율별 파일은 edge가 있는 shot만 포함. 146~147mm 평가 residual도 저장. 번호는 00번 제품 순서."],
], columns=["item","description"])
er_tables = {"Sampling_summary":er_sampling_summary,"Wafer_point_counts":er_wafer_counts,"Wafer_edge_shot_counts":er_edge_shot_counts,"Shot_point_counts":er_shot_counts,
    "Mixed_wafer_input_comparison":er_wafer_gaps,"Mixed_shot_input_comparison":er_shot_gaps,"Product_curves":er_product_curves,"Wafer_M3S":er_wafer_metrics,"Shot_M3S":er_shot_metrics,
    "Radius_delta":er_radius_delta,"Wafer_best_scale":er_best_scales,"Previous_opt_loss":er_reference_loss,"Shot_inner_M3S_wide":er_shot_inner_wide,"Diagnostic_summary":er_diagnostic_summary,
    "Baseline_parity":er_parity,"Input_audit":er_input_audit,"Input_files":pd.DataFrame(er_inventory),"Wafer_index":er_wafer_index,"Notes":er_notes}
for name,table in er_tables.items():
    table.to_parquet(EP_OUTPUT_DIR/"tables"/f"{name}.parquet",index=False)
excel_tables = {name:table for name,table in er_tables.items() if name not in ["Shot_M3S","Shot_point_counts"]}
excel_tables["Edge_shot_point_counts"] = er_shot_counts.loc[er_shot_counts.n_edge.gt(0)]


def er_excel_safe(table):
    out=table.copy()
    for col in out:
        if col.endswith("transn_seq") or col in ["apc_hist_index_no","apc_trocs_hist_index_no"]:
            out[col]=out[col].map(lambda value:str(int(value)) if pd.notna(value) else "")
    return out


er_excel_path=EP_OUTPUT_DIR/"OCO_Edge_Radius_Sweep.xlsx"
with pd.ExcelWriter(er_excel_path,engine="openpyxl") as writer:
    for name,table in excel_tables.items():
        table=er_excel_safe(table)
        for offset in range(0,max(1,len(table)),1048575):
            suffix="" if len(table)<=1048575 else f"_{offset//1048575+1}"
            table.iloc[offset:offset+1048575].to_excel(writer,sheet_name=name[:31-len(suffix)]+suffix,index=False)
er_settings=dict(sources=er_sources,fit_radii_mm=ER_FIT_RADII_MM,scales=ER_SCALES,inner_radius_um=EP_INNER_RADIUS_UM,evaluation_max_radius_um=EP_MAX_RADIUS_UM,std_ddof=EP_STD_DDOF,
    fixed_kmrc=True,metric_basis="UNSCALED_INPUT_MINUS_CPE",strict_prediction_coverage=True,complete_wafer_cohort_for_curves=True,save_scenario_points=ER_SAVE_SCENARIO_POINTS)
(EP_OUTPUT_DIR/"sweep_settings.json").write_text(json.dumps(er_settings,ensure_ascii=False,indent=2),encoding="utf-8")
display(FileLink(os.path.relpath(er_excel_path,Path.cwd()),result_html_prefix="회귀 반경·edge 배율 비교 Excel: "))
for path in er_plot_paths:
    if path.suffix==".png":
        display(FileLink(os.path.relpath(path,Path.cwd()),result_html_prefix="그래프: "))
print("분석 완료:",EP_OUTPUT_DIR)
